In [1]:
%pip -q install -U "timesfm[torch]" pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\users\\panst\\desktop\\ucl\\t3\\dissertation\\commodity-optionality-pricing-ucl-dissertation\\.venv\\lib\\site-packages\\matplotlib\\backends\\_backend_agg.cp311-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import numpy as np

def _detect_device():
    try:
        import torch
        return "gpu" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"

def _dedupe_and_sort_int_ts(t: np.ndarray, x: np.ndarray):
    """
    Sort by integer timestamps and collapse duplicates by averaging values.
    Returns unique_t (int64), mean_x (float64)
    """
    t = np.asarray(t).astype(np.int64)
    x = np.asarray(x).astype(np.float64)

    # keep only finite x
    m = np.isfinite(x)
    t, x = t[m], x[m]

    order = np.argsort(t)
    t, x = t[order], x[order]

    uniq_t, inv = np.unique(t, return_inverse=True)
    sums = np.bincount(inv, weights=x)
    counts = np.bincount(inv)
    means = sums / np.maximum(counts, 1)
    return uniq_t, means

def _regularize_integer_axis(t: np.ndarray, x: np.ndarray):
    """
    1) sort & de-duplicate,
    2) choose the most common positive step,
    3) fill a full integer grid with linear interpolation.
    Returns: t_reg (int64), x_reg (float32), step (int)
    """
    t, x = _dedupe_and_sort_int_ts(t, x)
    if len(t) <= 1:
        return t, x.astype(np.float32), 1

    diffs = np.diff(t)
    pos = diffs[diffs > 0]
    if pos.size == 0:
        step = 1
    else:
        vals, counts = np.unique(pos, return_counts=True)
        step = int(vals[np.argmax(counts)])

    t_full = np.arange(t[0], t[-1] + step, step, dtype=np.int64)
    x_full = np.interp(t_full, t, x).astype(np.float32)
    return t_full, x_full, step

def timesfm_forecast_from_arrays(
    t: np.ndarray,
    x: np.ndarray,
    horizon: int,
    freq_code: int = 0,  # 0=high (hour/day), 1=medium (week/month), 2=low (quarter/year)
    checkpoint: str = "google/timesfm-1.0-200m-pytorch",
    device: str | None = None,
):
    """
    Returns:
      future_t (int64 array), yhat (float32 array)
    """
    if device is None:
        device = _detect_device()

    # 1) regularize to even grid
    t_reg, x_reg, step = _regularize_integer_axis(t, x)

    # 2) load TimesFM
    import timesfm
    tfm = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(
            backend=device,
            per_core_batch_size=1,
            horizon_len=int(horizon),
        ),
        checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id=checkpoint),
    )

    # 3) forecast (model consumes values; freq is a category)
    point_forecast, _ = tfm.forecast([x_reg], freq=[int(freq_code)])
    yhat = np.asarray(point_forecast)[0].reshape(-1).astype(np.float32)

    # 4) future integer timestamps using detected step
    last_t = int(t_reg[-1])
    future_t = np.arange(last_t + step, last_t + step * (horizon + 1), step, dtype=np.int64)

    return future_t, yhat, t_reg, x_reg


In [ ]:
import pandas as pd

df = pd.read_csv('C:/Users/PanSt/Desktop/UCL/T3/Dissertation/Commodity-Optionality-Pricing-UCL-Dissertation/Data/aluminium_pre_inputs.csv')
df.head()

,date,al_lme_prices_log_returns,al_lme_prices_daily_returns,us_dollar_index,canadian_dollar_spot,emirate_dirham_spot,russian_ruble_spot,nor_krone_spot,australian_dollar_spot,malaysian_ringgit_spot,...,us_gdp_agr,china_caixin_pmi,germany_pmi,japan_pmi,us_pmi,1w_vol,1m_vol,3m_vol,1y_vol,al_lme_prices_abs_log_returns
0,1/18/2016,0.009106,0.009148,98.956,1.4561,3.6726,79.3775,8.8984,1.457089,4.3930,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.009106
1,1/19/2016,-0.004711,-0.004700,98.991,1.4576,3.6723,78.7850,8.8061,1.448646,4.3625,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.004711
2,1/20/2016,-0.012730,-0.012650,99.091,1.4501,3.6723,81.3475,8.8728,1.448016,4.3910,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.012730
3,1/21/2016,0.015257,0.015374,99.056,1.4261,3.6722,82.3675,8.7823,1.429388,4.3730,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.015257
4,1/22/2016,-0.002189,-0.002187,99.574,1.4115,3.6722,78.1000,8.7341,1.428571,4.2940,...,2.0,48.2,53.2,52.6,51.2,0.177532,NaN,NaN,NaN,0.002189


In [23]:
t = df.index[:-4].to_numpy()
len(t)

2441

In [24]:
x = df['1w_vol'][4:].values
len(x)

2441

In [25]:
HORIZON   = 5
FREQ_CODE = 0   # 0=high, 1=medium, 2=low
CHECKPOINT = "google/timesfm-1.0-200m-pytorch"  # or "google/timesfm-2.0-500m-pytorch"
DEVICE     = None  # None → auto (gpu if available)

future_t, yhat, t_reg, x_reg = timesfm_forecast_from_arrays(
    t, x, horizon=HORIZON, freq_code=FREQ_CODE, checkpoint=CHECKPOINT, device=DEVICE
)

print("Detected last observed t:", int(t_reg[-1]))
print("Next timestamps (first 10):", future_t[:10])
print("Forecast (first 10):", np.round(yhat[:10], 4))


OSError: [WinError 126] The specified module could not be found. Error loading "C:\Users\PanSt\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\lib\shm.dll" or one of its dependencies.